# Alpha Intelligence Engine — Model Training

**Goal:** Train Linear Regression (baseline) and XGBoost models to predict next-day stock alpha using Nifty 100 data from Yahoo Finance.

**Alpha Definition:**  
  `Alpha[t] = Stock_Return[t] − Rolling_Beta[t] × Market_Return[t]`  
  where `Rolling_Beta` is a 60-day rolling covariance/variance against NIFTY 50.

**Output:** Saved models (`.pkl`) for use by the Streamlit inference app.

In [ ]:
# ==========================================
# 1. IMPORTS & SETUP
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import pickle
import os
import warnings
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Setup complete.')

---
## 2. Configuration

In [ ]:
# ==========================================
# 2. CONFIGURATION
# ==========================================
NIFTY_100_STOCKS = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','ICICIBANK.NS','INFY.NS',
    'HINDUNILVR.NS','ITC.NS','KOTAKBANK.NS','LT.NS','AXISBANK.NS',
    'SBIN.NS','BHARTIARTL.NS','MARUTI.NS','HCLTECH.NS','ASIANPAINT.NS',
    'SUNPHARMA.NS','TITAN.NS','BAJFINANCE.NS','NTPC.NS','POWERGRID.NS',
    'M&M.NS','ULTRACEMCO.NS','WIPRO.NS','BAJAJFINSV.NS','ADANIPORTS.NS',
    'HDFCLIFE.NS','TECHM.NS','TATAMOTORS.NS','ONGC.NS','NESTLEIND.NS',
    'JSWSTEEL.NS','GRASIM.NS','TATASTEEL.NS','INDUSINDBK.NS','DIVISLAB.NS',
    'DRREDDY.NS','BRITANNIA.NS','APOLLOHOSP.NS','CIPLA.NS','HINDALCO.NS',
    'COALINDIA.NS','SBILIFE.NS','EICHERMOT.NS','BAJAJ-AUTO.NS','ADANIENT.NS',
    'HEROMOTOCO.NS','TATACONSUM.NS','SHRIRAMFIN.NS','TRENT.NS','BEL.NS',
    'VEDL.NS','IOC.NS','BPCL.NS','HINDZINC.NS','PIDILITIND.NS',
    'DLF.NS','SIEMENS.NS','LTIM.NS','BAJAJHLDNG.NS','ICICIPRULI.NS',
    'TVSMOTOR.NS','MARICO.NS','HAVELLS.NS','DABUR.NS','AMBUJACEM.NS',
    'ICICIGI.NS','TORNTPHARM.NS','MCDOWELL-N.NS','SRTRANSFIN.NS',
    'BANKBARODA.NS','INDIGO.NS','ZOMATO.NS','PAGEIND.NS','DMART.NS',
    'PERSISTENT.NS','COLPAL.NS','POLYCAB.NS','ATGL.NS','ADANIENSOL.NS',
]

NIFTY_INDEX = '^NSEI'
DATA_DIR = '../data'
MODELS_DIR = '../models'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

FEATURE_COLS = [
    'RSI', 'Volatility', 'Price_to_SMA_50', 'Price_to_SMA_20',
    'SMA_Crossover', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_3',
    'Return_Lag_5', 'Return_Momentum_5', 'Return_Momentum_10',
    'Rolling_Beta', 'Market_Return_Lag_1'
]

print(f'Configured: {len(NIFTY_100_STOCKS)} stocks, {len(FEATURE_COLS)} features')

---
## 3. Data Fetching

Download historical data for all Nifty 100 stocks + the NIFTY 50 index from Yahoo Finance.

In [ ]:
# ==========================================
# 3. DATA FETCHING
# ==========================================
end_date = datetime.now()
start_date = end_date - timedelta(days=730)

print('Downloading NIFTY 50 index data...')
nifty = yf.download(NIFTY_INDEX, start=start_date, end=end_date, progress=True)
if isinstance(nifty.columns, pd.MultiIndex):
    nifty = nifty.droplevel(1, axis=1)
nifty['Market_Return'] = np.log(nifty['Close'] / nifty['Close'].shift(1))
nifty.to_csv(f'{DATA_DIR}/nifty_index.csv')
print(f'NIFTY data: {len(nifty)} rows, {nifty.index[0].date()} to {nifty.index[-1].date()}')

all_data = {}
failed = []
total = len(NIFTY_100_STOCKS)

for i, ticker in enumerate(NIFTY_100_STOCKS):
    print(f'[{i+1}/{total}] Fetching {ticker}...', end=' ')
    try:
        data = yf.download(ticker, start=start_date, end=end_date, progress=False)
        if data.empty:
            print('EMPTY')
            failed.append(ticker)
            continue
        if isinstance(data.columns, pd.MultiIndex):
            data = data.droplevel(1, axis=1)
        if 'Close' not in data.columns or len(data) < 100:
            print('INSUFFICIENT DATA')
            failed.append(ticker)
            continue
        all_data[ticker] = data
        print(f'{len(data)} rows')
    except Exception as e:
        print(f'ERROR: {e}')
        failed.append(ticker)

print(f'\nSuccessfully fetched: {len(all_data)}/{total} stocks')
print(f'Failed: {len(failed)} stocks')
if failed:
    print(f'Failed tickers: {failed}')

---
## 4. Feature Engineering & Alpha Calculation

For each stock:
1. Calculate technical indicators (RSI, volatility, SMA crossovers, lagged returns, momentum)
2. Calculate rolling beta (60-day) against NIFTY 50
3. Calculate alpha: `Alpha[t] = Return[t] − Beta[t] × Market_Return[t]`
4. Target: `Alpha[t+1]` (next-day alpha)

In [ ]:
# ==========================================
# 4. FEATURE ENGINEERING & ALPHA CALCULATION
# ==========================================
processed_dfs = []

for ticker, data in all_data.items():
    df = data.copy()
    name = ticker.replace('.NS', '')

    df['Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    df['SMA_20'] = df['Close'].rolling(window=20).mean()
    df['Price_to_SMA_50'] = df['Close'] / df['SMA_50']
    df['Price_to_SMA_20'] = df['Close'] / df['SMA_20']
    df['SMA_Crossover'] = df['SMA_20'] / df['SMA_50']

    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    df['RSI'] = 100 - (100 / (1 + (gain / (loss + 1e-9))))
    df['Volatility'] = df['Returns'].rolling(window=21).std()

    for lag in [1, 2, 3, 5]:
        df[f'Return_Lag_{lag}'] = df['Returns'].shift(lag)
    df['Return_Momentum_5'] = df['Returns'].rolling(5).mean()
    df['Return_Momentum_10'] = df['Returns'].rolling(10).mean()

    merged = df.join(nifty[['Market_Return']], how='inner')
    merged['Market_Return_Lag_1'] = merged['Market_Return'].shift(1)

    cov = merged['Returns'].rolling(60).cov(merged['Market_Return'])
    var = merged['Market_Return'].rolling(60).var() + 1e-9
    merged['Rolling_Beta'] = cov / var
    merged['Rolling_Beta'] = merged['Rolling_Beta'].clip(-2, 5)

    merged['Alpha'] = merged['Returns'] - merged['Rolling_Beta'] * merged['Market_Return']
    merged['Target_Alpha'] = merged['Alpha'].shift(-1)
    merged['Ticker'] = name

    processed_dfs.append(merged.dropna())

pooled_df = pd.concat(processed_dfs, ignore_index=True)
print(f'Pooled dataset: {len(pooled_df):,} rows, {pooled_df["Ticker"].nunique()} unique stocks')
print(f'Columns: {list(pooled_df.columns)}')

In [ ]:
# Save raw processed data
pooled_df.to_csv(f'{DATA_DIR}/pooled_stock_data.csv', index=False)
print('Saved pooled data to ../data/pooled_stock_data.csv')

---
## 5. Exploratory Data Analysis (EDA)

In [ ]:
# ==========================================
# 5. EXPLORATORY DATA ANALYSIS
# ==========================================
df_clean = pooled_df.dropna(subset=FEATURE_COLS + ['Target_Alpha']).copy()

print(f'Clean dataset: {len(df_clean):,} rows')
print(f'\nTarget (Alpha) statistics:')
print(df_clean['Alpha'].describe())
print(f'\nTarget_Alpha statistics:')
print(df_clean['Target_Alpha'].describe())

In [ ]:
# Distribution of Alpha
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_clean['Alpha'], bins=80, color='#3b82f6', edgecolor='white', alpha=0.7)
axes[0].set_title('Historical Alpha Distribution')
axes[0].set_xlabel('Alpha')
axes[0].set_ylabel('Frequency')

axes[1].hist(df_clean['Target_Alpha'], bins=80, color='#22c55e', edgecolor='white', alpha=0.7)
axes[1].set_title('Target Alpha (next-day) Distribution')
axes[1].set_xlabel('Target Alpha')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/eda_alpha_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature-target correlations
corr_data = df_clean[FEATURE_COLS + ['Target_Alpha']].corr()['Target_Alpha'].drop('Target_Alpha').sort_values()

plt.figure(figsize=(10, 6))
colors = ['#ef4444' if v < 0 else '#22c55e' for v in corr_data.values]
plt.barh(range(len(corr_data)), corr_data.values, color=colors)
plt.yticks(range(len(corr_data)), corr_data.index)
plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Correlation with Target Alpha')
plt.title('Feature Correlations with Next-Day Alpha')
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/eda_feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFeature correlations with Target Alpha:')
for feat, corr in corr_data.items():
    print(f'  {feat:25s}: {corr:+.4f}')

In [ ]:
# Feature correlation heatmap
plt.figure(figsize=(14, 10))
corr_matrix = df_clean[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Alpha by stock (boxplot of top/bottom)
stock_means = df_clean.groupby('Ticker')['Alpha'].mean().sort_values()
top_bottom = pd.concat([stock_means.head(10), stock_means.tail(10)])

plt.figure(figsize=(12, 5))
top_bottom.plot(kind='bar', color=['#ef4444']*10 + ['#22c55e']*10)
plt.title('Stocks with Highest/Lowest Mean Alpha')
plt.ylabel('Mean Alpha')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/eda_stock_alpha.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Model Training

Split: Time-ordered 80/20 (no look-ahead).  
Models:
- **Linear Regression** — baseline
- **XGBoost** — gradient boosting

In [ ]:
# ==========================================
# 6. MODEL TRAINING
# ==========================================
data_ml = df_clean[FEATURE_COLS + ['Target_Alpha']].dropna().copy()
print(f'ML dataset: {len(data_ml):,} rows')

X = data_ml[FEATURE_COLS].values
y = data_ml['Target_Alpha'].values

split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'y_train mean: {y_train.mean():.6f}  std: {y_train.std():.6f}')
print(f'y_test  mean: {y_test.mean():.6f}  std: {y_test.std():.6f}')

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Scaler fitted. Mean: {scaler.mean_[:3]}...  Std: {scaler.scale_[:3]}...')

In [ ]:
# Train Linear Regression (Baseline)
print('Training Linear Regression (baseline)...')
lr = LinearRegression()
lr.fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)
print(f'  Test R²:  {r2_score(y_test, lr_pred):.4f}')
print(f'  Test MAE: {mean_absolute_error(y_test, lr_pred):.6f}')

print('\nLinear Regression Coefficients:')
for feat, coef in sorted(zip(FEATURE_COLS, lr.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {feat:25s}: {coef:+.6f}')

In [ ]:
# Train XGBoost
print('Training XGBoost...')
xgb = XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42, verbosity=0)
xgb.fit(X_train_s, y_train)
xgb_pred = xgb.predict(X_test_s)
print(f'  Test R²:  {r2_score(y_test, xgb_pred):.4f}')
print(f'  Test MAE: {mean_absolute_error(y_test, xgb_pred):.6f}')

In [ ]:
# XGBoost Feature Importance
importance = xgb.feature_importances_
imp_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importance})
imp_df = imp_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(range(len(imp_df)), imp_df['Importance'].values, color='#3b82f6')
plt.yticks(range(len(imp_df)), imp_df['Feature'].values)
plt.xlabel('Feature Importance')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/xgb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Model Evaluation

In [ ]:
# ==========================================
# 7. MODEL EVALUATION
# ==========================================
lr_r2 = r2_score(y_test, lr_pred)
xgb_r2 = r2_score(y_test, xgb_pred)
lr_mae = mean_absolute_error(y_test, lr_pred)
xgb_mae = mean_absolute_error(y_test, xgb_pred)

print('='*50)
print('MODEL COMPARISON')
print('='*50)
print(f'{"Model":30s} {"R²":>8s} {"MAE":>12s}')
print('-'*50)
print(f'{"Linear Regression (Baseline)":30s} {lr_r2:>8.4f} {lr_mae:>12.6f}')
print(f'{"XGBoost":30s} {xgb_r2:>8.4f} {xgb_mae:>12.6f}')
print('-'*50)

if xgb_r2 >= lr_r2:
    print(f'\n Best Model: XGBoost (R² = {xgb_r2:.4f})')
    best_model = xgb
    best_name = 'XGBoost'
else:
    print(f'\n Best Model: Linear Regression (R² = {lr_r2:.4f})')
    best_model = lr
    best_name = 'LinearRegression'

In [ ]:
# Actual vs Predicted plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, lr_pred, alpha=0.3, s=5, color='#3b82f6')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', alpha=0.5)
axes[0].set_xlabel('Actual Alpha')
axes[0].set_ylabel('Predicted Alpha')
axes[0].set_title(f'Linear Regression (R² = {lr_r2:.4f})')

axes[1].scatter(y_test, xgb_pred, alpha=0.3, s=5, color='#22c55e')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', alpha=0.5)
axes[1].set_xlabel('Actual Alpha')
axes[1].set_ylabel('Predicted Alpha')
axes[1].set_title(f'XGBoost (R² = {xgb_r2:.4f})')

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/model_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(y_test - lr_pred, bins=60, color='#3b82f6', alpha=0.7)
axes[0].set_title(f'LR Residuals (std: {(y_test - lr_pred).std():.6f})')
axes[0].set_xlabel('Residual')

axes[1].hist(y_test - xgb_pred, bins=60, color='#22c55e', alpha=0.7)
axes[1].set_title(f'XGBoost Residuals (std: {(y_test - xgb_pred).std():.6f})')
axes[1].set_xlabel('Residual')

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/model_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Interpretation of Results

**Important:** Predicting next-day alpha is fundamentally difficult because:
- Alpha is the residual after removing market beta — it's designed to be the "unpredictable" component
- In efficient markets, alpha signals get arbitraged away
- Most of the predictable component in returns is captured by beta/market factors

**Expected R² range:**
- R² ≈ 0.00 to 0.05 is realistic for next-day alpha prediction
- Negative R² means the model is worse than predicting the mean (common with small datasets)
- The value is in the ranking (relative ordering), not the absolute prediction

**Why this still adds value:**
- Even low R² can produce a ranking that separates high-alpha from low-alpha stocks
- A portfolio of top-quintile predicted alpha stocks can outperform bottom-quintile
- The feature importance tells us which signals have edge

---
## 9. Save Models for Inference

In [ ]:
# ==========================================
# 9. SAVE MODELS
# ==========================================
model_artifacts = {
    'scaler': scaler,
    'linear_regression': lr,
    'xgboost': xgb,
    'feature_cols': FEATURE_COLS,
    'lr_r2': float(lr_r2),
    'xgb_r2': float(xgb_r2),
    'best_model_name': best_name,
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'n_stocks': len(all_data),
    'n_training_samples': len(X_train),
}

with open(f'{MODELS_DIR}/alpha_model.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print(f'Models saved to {MODELS_DIR}/alpha_model.pkl')
print(f'\nArtifacts included:')
for key in model_artifacts:
    val = model_artifacts[key]
    if isinstance(val, (int, float, str)):
        print(f'  {key}: {val}')
    else:
        print(f'  {key}: {type(val).__name__}')

In [ ]:
# Verify the saved model works
print('Verifying saved model...')
with open(f'{MODELS_DIR}/alpha_model.pkl', 'rb') as f:
    loaded = pickle.load(f)

print(f'  Loaded scaler: {type(loaded["scaler"]).__name__}')
print(f'  Loaded LR: {type(loaded["linear_regression"]).__name__}')
print(f'  Loaded XGB: {type(loaded["xgboost"]).__name__}')
print(f'  Best model: {loaded["best_model_name"]}')
print(f'  Training date: {loaded["training_date"]}')
print(f'  Feature cols: {len(loaded["feature_cols"])} features')

# Quick inference test
test_sample = X_test_s[:1]
lr_pred_test = loaded['linear_regression'].predict(test_sample)
xgb_pred_test = loaded['xgboost'].predict(test_sample)
print(f'\nSample prediction:')
print(f'  Actual:       {y_test[0]:.6f}')
print(f'  LR predict:   {lr_pred_test[0]:.6f}')
print(f'  XGB predict:  {xgb_pred_test[0]:.6f}')
print('\nVerification complete.')

---
## Summary

**What was done:**
1. Fetched Nifty 100 stock data + NIFTY 50 index from Yahoo Finance
2. Engineered 13 features: technical indicators, lagged returns, momentum, rolling beta
3. Calculated true alpha: `Return − Beta × Market Return`
4. Performed EDA: distributions, correlations, stock-level alpha analysis
5. Trained Linear Regression (baseline) and XGBoost on time-ordered 80/20 split
6. Compared performance and saved the best model

**Saved to `../models/alpha_model.pkl`:**
- StandardScaler
- LinearRegression model
- XGBoost model
- Feature column names
- Performance metrics

**Next:** Run the Streamlit app — it will load these models for inference instead of training.